# Fannie Mae vs. Freddie Mac: do their loans prepay the same way?

We compare a **sample** of Fannie Mae and Freddie Mac loans. If both prepay the same way once the data is put in the same format,
we can train one model on both, with an `agency` column.

In [1]:
import sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "source"))
import fannie_load_data as fannie
import freddie_load_data as freddie

FANNIE_CACHE = ROOT / "data" / "fannie" / "processed" / "perf_sample_80mb.parquet"
FREDDIE_CACHE = ROOT / "data" / "freddie" / "processed" / "perf_sample_30mb.parquet"  # 30 MB/file ≈ same loan count as Fannie's 80 MB
VINTAGES = (2000, 2025)  # origination years in both datasets
MIN_LOAN_MONTHS = 2_000  # hide buckets with less data than this

# build the samples on first run only
if not FANNIE_CACHE.exists():
    fannie.load_performance_sample(FANNIE_CACHE, mb_per_file=80)
if not FREDDIE_CACHE.exists():
    freddie.load_performance_sample(FREDDIE_CACHE, mb_per_file=30)

COLOR = {"fannie": "#2a78d6", "freddie": "#eb6834"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e5e0", "axes.axisbelow": True, "legend.frameon": False})

## 1. Freddie data checks

Quick sanity checks on the Freddie sample. Each line is the share of rows that pass (1.0 = all).

In [2]:
def month_index(col, fmt):
    # date string -> months since year 0 (Freddie uses YYYYMM, Fannie MMYYYY)
    y, m = (pl.col(col).str.slice(0, 4), pl.col(col).str.slice(4, 2)) if fmt == "YYYYMM" else (pl.col(col).str.slice(2, 4), pl.col(col).str.slice(0, 2))
    return y.cast(pl.Int32) * 12 + m.cast(pl.Int32) - 1

fr = pl.scan_parquet(FREDDIE_CACHE).with_columns(t=month_index("rep_period", "YYYYMM"), fpd=month_index("first_pay_date", "YYYYMM"))

fr.select(
    matched_orig_record=pl.col("orig_rate").is_not_null().mean(),
    rate_in_0_15=pl.col("orig_rate").is_between(0.01, 15).mean(),
    term_360=(pl.col("orig_term") == 360).mean(),
    fico_300_850=pl.col("fico").drop_nulls().is_between(300, 850).mean(),
    fico_missing=pl.col("fico").is_null().mean(),
    ltv_1_105=pl.col("ltv").drop_nulls().is_between(1, 105).mean(),  # HARP refis can go higher
    dti_0_65=pl.col("dti").drop_nulls().is_between(0, 65).mean(),
    loan_age_consistent=(pl.col("loan_age") == pl.col("t") - pl.col("fpd") + 1).drop_nulls().mean(),  # modified loans restart age
).collect().transpose(include_header=True, header_name="check", column_names=["share"])

check,share
str,f64
"""matched_orig_record""",1.0
"""rate_in_0_15""",1.0
"""term_360""",0.640664
"""fico_300_850""",1.0
"""fico_missing""",0.001672
"""ltv_1_105""",0.995998
"""dti_0_65""",1.0
"""loan_age_consistent""",0.989626


## 2. One combined table

Both samples get the same columns and codes, plus an `agency` column:

- **30-year loans** originated 2000–2025 only.
- **Prepay** = the loan was paid off in full this month (zero-balance code `01`).
- **Balance** used to weight speeds: original balance in the first 6 months (Fannie hides it, Freddie rounds it), else current balance.
- **Loan purpose:** Freddie's `N` (no-cash-out refi) becomes `R`, as in Fannie. Freddie's `R` ("refi, not specified") becomes missing.

In [3]:
COLS = ["agency", "loan_id", "t", "od", "age", "orig_rate", "orig_upb", "ltv", "dti", "fico", "num_borrowers",
        "first_time_buyer", "loan_purpose", "property_type", "occupancy", "channel", "prepay", "exposure", "prepaid"]
CAT = ["first_time_buyer", "loan_purpose", "property_type", "occupancy", "channel"]

def to_common(lf):
    return (lf.with_columns(od=pl.col("fpd") - 1, age=pl.col("t") - pl.col("fpd") + 1,
                            prepay=pl.col("zero_bal_code").cast(pl.Utf8) == "01")
              .filter(pl.col("orig_term") == 360, pl.col("age") >= 1, (pl.col("od") // 12).is_between(*VINTAGES))
              .with_columns(exposure=pl.when(pl.col("upb_at_removal").is_not_null()).then(pl.col("upb_at_removal"))
                                       .when((pl.col("age") <= 6) | ~(pl.col("cur_upb") > 0)).then(pl.col("orig_upb"))
                                       .otherwise(pl.col("cur_upb")))
              .with_columns(prepaid=pl.when(pl.col("prepay")).then(pl.col("exposure")).otherwise(0.0),
                            *[pl.col(c).cast(pl.Utf8) for c in CAT],
                            *[pl.col(c).cast(pl.Int32) for c in ["ltv", "dti", "fico", "num_borrowers"]])
              .select(COLS))

fannie_lf = pl.scan_parquet(FANNIE_CACHE).with_columns(
    agency=pl.lit("fannie"), t=month_index("rep_period", "MMYYYY"), fpd=month_index("first_pay_date", "MMYYYY"),
    fico=pl.coalesce("fico", "fico_classic_orig"))
freddie_lf = fr.with_columns(
    agency=pl.lit("freddie"), loan_purpose=pl.col("loan_purpose").cast(pl.Utf8).replace({"R": None, "N": "R"}))

panel = pl.concat([to_common(fannie_lf), to_common(freddie_lf)]).collect()
panel.group_by("agency").agg(loans=pl.col("loan_id").n_unique(), rows=pl.len(), payoffs=pl.col("prepay").sum()).sort("agency")

agency,loans,rows,payoffs
str,u32,u32,u32
"""fannie""",549017,21493304,306469
"""freddie""",446648,18544290,244391


**Refi incentive** = the loan's rate minus today's typical rate, so a positive value means the borrower could save by refinancing.
The "typical rate" is the average rate on new 30-year loans in our data (both agencies) last month.

In [4]:
new_loans = panel.group_by("agency", "loan_id").agg(pl.col("od", "orig_rate", "orig_upb").first())
mkt = (new_loans.group_by("od")
       .agg(mkt_rate=(pl.col("orig_rate") * pl.col("orig_upb")).sum() / pl.col("orig_upb").sum(), n=pl.len())
       .filter(pl.col("n") >= 30))
all_months = pl.DataFrame({"od": np.arange(mkt["od"].min(), panel["t"].max() + 1, dtype=np.int32)})
mkt = all_months.join(mkt, on="od", how="left").with_columns(pl.col("mkt_rate").interpolate().forward_fill())

panel = (panel.join(mkt.select(t=pl.col("od") + 1, mkt_rate="mkt_rate"), on="t", how="left")  # last month's rate
              .with_columns(incentive=pl.col("orig_rate") - pl.col("mkt_rate")))

In [5]:
def cut(col, breaks, labels):
    # left-closed bins, kept in their natural order
    return pl.col(col).cast(pl.Float64).bin_intervals([float(b) for b in breaks], labels=labels)

inc_breaks = [-1.0, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0, 1.5]
panel = panel.with_columns(
    inc_b=cut("incentive", inc_breaks, ["<-1", "-1 to -0.5", "-0.5 to -0.25", "-0.25 to 0", "0 to 0.25",
                                         "0.25 to 0.5", "0.5 to 0.75", "0.75 to 1", "1 to 1.5", "1.5+"]),
    age_b=cut("age", [13, 25, 37, 61, 121], ["1-12", "13-24", "25-36", "37-60", "61-120", "121+"]),
    fico_b=cut("fico", [660, 700, 740, 780], ["<660", "660-699", "700-739", "740-779", "780+"]),
    ltv_b=cut("ltv", [60.01, 75.01, 80.01, 90.01], ["<=60", "60-75", "75-80", "80-90", ">90"]),
    dti_b=cut("dti", [20, 30, 36, 43], ["<20", "20-29", "30-35", "36-42", "43+"]),
    upb_b=cut("orig_upb", [1e5, 2e5, 3e5, 4e5], ["<100k", "100-200k", "200-300k", "300-400k", "400k+"]),
    vintage=pl.col("od") // 12,
)

def cpr(frame, by, min_lm=MIN_LOAN_MONTHS):
    # annualized prepayment speed (%), weighted by balance; hidden where there is too little data
    return (frame.filter(pl.all_horizontal(pl.col(c).is_not_null() for c in by))
            .group_by(by).agg(smm=pl.col("prepaid").sum() / pl.col("exposure").sum(), loan_months=pl.len())
            .with_columns(cpr=pl.when(pl.col("loan_months") >= min_lm).then(100 * (1 - (1 - pl.col("smm")) ** 12)))
            .sort(by))

cpr(panel, ["agency"])

agency,smm,loan_months,cpr
str,f64,u32,f64
"""fannie""",0.014165,21493304,15.734326
"""freddie""",0.013022,18544290,14.554839


## 3. Coverage and speed over time

Left: do both samples hold loans from the same years? Right: monthly prepayment speed (CPR) for each agency.

In [6]:
cov = (panel.group_by("agency", "loan_id").agg(pl.col("vintage").first())
            .group_by("agency", "vintage").len()
            .with_columns(share=100 * pl.col("len") / pl.col("len").sum().over("agency")))
ts = cpr(panel, ["agency", "t"], min_lm=500).with_columns(date=pl.date(pl.col("t") // 12, pl.col("t") % 12 + 1, 1))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4), width_ratios=[1, 1.6])
for k, ag in enumerate(COLOR):
    d = cov.filter(pl.col("agency") == ag).sort("vintage")
    a1.bar(d["vintage"] + (k - 0.5) * 0.4, d["share"], width=0.4, color=COLOR[ag], label=ag.title())
    d = ts.filter(pl.col("agency") == ag)
    a2.plot(d["date"], d["cpr"], color=COLOR[ag], lw=1.4, label=ag.title())
a1.set(title="Share of loans by origination year", ylabel="% of loans")
a2.set(title="Monthly CPR", ylabel="CPR (%)")
a1.legend(); a2.legend()
fig.tight_layout()

## 4. Speed by loan characteristic

Same cuts as notebook 01, one line per agency. Lines on top of each other = same behavior.
A steady gap is fine (the `agency` column can learn it). A gap that changes shape would be a problem.

In [7]:
ONE_D = [("inc_b", "Refi incentive (pp)"), ("age_b", "Loan age (months)"), ("fico_b", "FICO"),
         ("ltv_b", "LTV (%)"), ("dti_b", "DTI (%)"), ("upb_b", "Loan size")]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (col, title) in zip(axes.flat, ONE_D):
    tbl = cpr(panel, ["agency", col])
    for ag in COLOR:
        d = tbl.filter(pl.col("agency") == ag)
        ax.plot(d[col].cast(pl.Utf8), d["cpr"], marker="o", color=COLOR[ag], label=ag.title())
    ax.set(title=title, ylabel="CPR (%)", ylim=(0, None))
    ax.tick_params(axis="x", rotation=45)
axes[0, 0].legend()
fig.tight_layout()

In [8]:
CATS = [("loan_purpose", "Loan purpose"), ("occupancy", "Occupancy"), ("channel", "Channel (T = Freddie only)"),
        ("property_type", "Property type"), ("first_time_buyer", "First-time buyer")]

fig, axes = plt.subplots(1, 5, figsize=(17, 3.5))
for ax, (col, title) in zip(axes, CATS):
    tbl = cpr(panel, ["agency", col]).filter(pl.col("cpr").is_not_null())
    cats = tbl[col].unique().sort().to_list()
    for k, ag in enumerate(COLOR):
        d = tbl.filter(pl.col("agency") == ag)
        ax.bar([cats.index(v) + (k - 0.5) * 0.4 for v in d[col]], d["cpr"], width=0.4, color=COLOR[ag], label=ag.title())
    ax.set_xticks(range(len(cats)), cats)
    ax.set(title=title, ylabel="CPR (%)")
axes[0].legend()
fig.tight_layout()

## 5. Refi incentive curve by loan age

Incentive is the main driver of prepayment, so we check its curve within each loan-age group.

In [9]:
panel = panel.with_columns(age_4=cut("age", [13, 37, 61], ["1-12", "13-36", "37-60", "61+"]))
tbl = cpr(panel, ["agency", "age_4", "inc_b"])
groups = ["1-12", "13-36", "37-60", "61+"]

fig, axes = plt.subplots(1, 4, figsize=(17, 3.8), sharey=True)
for ax, g in zip(axes, groups):
    for ag in COLOR:
        d = tbl.filter(pl.col("agency") == ag, pl.col("age_4") == g)
        ax.plot(d["inc_b"].cast(pl.Utf8), d["cpr"], marker="o", color=COLOR[ag], label=ag.title())
    ax.set(title=f"Loan age {g} months", xlabel="Refi incentive (pp)")
    ax.tick_params(axis="x", rotation=60)
axes[0].set_ylabel("CPR (%)"); axes[0].legend()
fig.tight_layout()

## 6. Speed after matching similar loans

Raw curves can differ just because the samples hold different loans. Here each agency's payoffs are compared with
what we'd expect from **similar loans of both agencies** (same month, incentive, age, FICO and LTV).
1.0 = as fast as similar loans; 0.95 = 5% slower.

In [10]:
panel = panel.with_columns(fico_4=cut("fico", [680, 740, 780], ["<680", "680-739", "740-779", "780+"]),
                           ltv_4=cut("ltv", [60.01, 80.01, 90.01], ["<=60", "60-80", "80-90", ">90"]),
                           year=pl.col("t") // 12)
SIMILAR = ["t", "inc_b", "age_4", "fico_4", "ltv_4"]

cells_ = (panel.drop_nulls(SIMILAR)
          .group_by(SIMILAR + ["agency", "year"]).agg(exp=pl.col("exposure").sum(), pre=pl.col("prepaid").sum())
          .with_columns(expected=pl.col("exp") * pl.col("pre").sum().over(SIMILAR) / pl.col("exp").sum().over(SIMILAR)))

def rel_speed(by):
    return cells_.group_by(by).agg(rel_speed=pl.col("pre").sum() / pl.col("expected").sum()).sort(by)

by_year = rel_speed(["agency", "year"])
fig, ax = plt.subplots(figsize=(11, 3.5))
for ag in COLOR:
    d = by_year.filter(pl.col("agency") == ag)
    ax.plot(d["year"], d["rel_speed"], marker="o", color=COLOR[ag], label=ag.title())
ax.axhline(1, color="grey", lw=1)
ax.set(title="Speed vs. similar loans, by year", ylabel="actual / expected")
ax.legend()
rel_speed(["agency"])

agency,rel_speed
str,f64
"""fannie""",1.018237
"""freddie""",0.975034


## 7. Takeaways

- **Freddie data is clean.** Every loan matched its origination record, FICO is missing on 0.2% of rows, and loan age is consistent on 99% of rows (the rest are modified loans).
- **Same years covered.** Both samples hold loans from every origination year 2000–2025 in similar proportions (about 550k Fannie and 450k Freddie loans).
- **Same patterns.** Speeds over time follow the same refinancing waves (2003, 2009, 2012, 2020–21). The incentive, age, FICO, LTV, DTI and loan-size curves have the same shape for both agencies.
- **Freddie is a bit slower:** 14.6% vs. 15.7% CPR overall, a steady gap rather than a different shape. After matching similar loans, Fannie runs at 1.02 and Freddie at 0.98.
- **The gap moves over time**: Freddie was clearly slower in 2004–07 and 2011–18, and faster in 2008–09. Since 2019 the two are close.
- **Watch-outs:** Freddie's channel `T` (third party, not specified) has no Fannie match and prepays fast (25% CPR). Fannie has an unknown purpose `U` that Freddie lacks.

**Decision:** train one model on both datasets with an `agency` column. The `agency` column, plus the reporting month, should absorb the small, time-varying gap.